# 02 — AI Scoring + Gold Scorecard

**What this does:** Runs AI functions on each stitched conversation to produce a complete QA scorecard.

| AI Function           | What It Produces                                              |
|-----------------------|--------------------------------------------------------------|
| `ai_analyze_sentiment()` | Sentiment label (Positive/Negative/Neutral/Mixed)            |
| `ai_classify()`          | Call category + disposition + protocol adherence             |
| `ai_similarity()`        | Script adherence scores (greeting/closing vs expected)       |
| `ai_query()`             | 10-criterion rubric scores (1-5) + coaching notes            |
| `ai_summarize()`         | 20-word call summaries for dashboard                         |
| `ai_mask()`              | HIPAA-compliant PII-redacted transcripts                     |

**Rubric Criteria 1, 2, 3:** E2E Functionality + Business Impact + AI Quality

---
*Prerequisite: Run `00_Generate_Data` then `01_Stitch_Transcripts` first*

In [0]:
%sql
-- Analyze customer sentiment for each call
CREATE OR REPLACE TEMP VIEW scored_sentiment AS
SELECT 
  interaction_id,
  agent_name,
  queue,
  direction,
  division,
  full_transcript,
  ai_analyze_sentiment(full_transcript) AS sentiment,
  CAST(transcript_duration AS INT) AS call_duration_seconds,
  num_utterances
FROM mmt_aws_usw2_catalog.contact_calls.silver_conversations

In [0]:
%sql
-- Compare agent's actual greeting/closing against expected scripts using ai_similarity
-- UPDATE: Replace 'HCP' below with your HC_PROVIDER value from notebook 00 config
-- Expected scripts:
--   Greeting: 'Thank you for calling HCP, this is [agent name] speaking. How may I help you today?'
--   Closing: 'Is there anything else I can help you with? Thank you for calling HCP. Have a wonderful day.'
-- Scores 0.0 to 1.0: how close the agent's language matches the ideal
CREATE OR REPLACE TEMP VIEW scored_classified AS
WITH base_classified AS (
  SELECT 
    s.*,
    ai_classify(full_transcript, ARRAY('Billing Dispute', 'Appointment Scheduling', 'Clinical Triage', 'Complaint', 'General Inquiry', 'Insurance Question', 'Prescription Refill')) AS call_category,
    ai_classify(full_transcript, ARRAY('escalate_to_supervisor', 'routine', 'coaching_opportunity')) AS disposition,
    ai_classify(full_transcript, ARRAY('compliant', 'partially_compliant', 'non_compliant')) AS protocol_adherence
  FROM scored_sentiment s
),
-- Extract first and last agent utterances for script adherence check
agent_lines AS (
  SELECT 
    interaction_id,
    -- First agent line = greeting
    REGEXP_EXTRACT(full_transcript, '(?m)^\\[.*?\\] INTERNAL: (.+?)$', 1) AS actual_greeting,
    -- Last agent line = closing
    REGEXP_EXTRACT(full_transcript, '(?m).*\\[.*?\\] INTERNAL: (.+?)$', 1) AS actual_closing
  FROM mmt_aws_usw2_catalog.contact_calls.silver_conversations
)
SELECT
  c.*,
  ai_similarity(
    a.actual_greeting,
    'Thank you for calling HCP, this is [agent name] speaking. How may I help you today?'
  ) AS greeting_adherence_score,
  ai_similarity(
    a.actual_closing,
    'Is there anything else I can help you with? Thank you for calling HCP. Have a wonderful day.'
  ) AS closing_adherence_score
FROM base_classified c
LEFT JOIN agent_lines a ON c.interaction_id = a.interaction_id

In [0]:
%sql
-- Score each call against the QA rubric using ai_query
-- Rubric sourced from mmt_aws_usw2_catalog.contact_calls.qa_rubric (10 criteria)
CREATE OR REPLACE TEMP VIEW gold_qa_evaluations AS
WITH rubric_text AS (
  SELECT CONCAT_WS('\n', COLLECT_LIST(
    CONCAT('- ', criterion, ' (weight ', CAST(weight AS STRING), '): ', expected_behavior)
  )) AS rubric_prompt
  FROM mmt_aws_usw2_catalog.contact_calls.qa_rubric
)
SELECT 
  c.*,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Score this contact center call transcript on the following criteria (1=poor, 5=exceptional). ',
      'Return ONLY a JSON object with keys: ',
      'greeting_score, identity_verification_score, empathy_score, commitment_score, ',
      'branding_score, compliance_score, resolution_score, further_assistance_score, ',
      'closing_score, customer_service_score, ',
      'overall_score (weighted average using weights below), coaching_notes (one sentence), requires_human_review (true ONLY if overall_score < 3.0 or a critical safety/compliance violation occurred, false otherwise). ',
      '\n\nRubric:\n', r.rubric_prompt,
      '\n\nTranscript:\n', c.full_transcript
    )
  ) AS qa_scores_json
FROM scored_classified c
CROSS JOIN rubric_text r

In [0]:
%sql
-- Parse JSON scores + script adherence → persist gold_scorecard for notebooks 03, 04, 05
-- LLM wraps JSON in code fences; extract the JSON object between first { and last }
CREATE OR REPLACE TABLE mmt_aws_usw2_catalog.contact_calls.gold_scorecard AS
WITH cleaned AS (
  SELECT *,
    REGEXP_EXTRACT(qa_scores_json, '(?s)(\\{.*\\})', 1) AS clean_json
  FROM gold_qa_evaluations
  WHERE qa_scores_json IS NOT NULL AND qa_scores_json != ''
)
SELECT
  interaction_id,
  agent_name,
  queue,
  direction,
  division,
  call_duration_seconds,
  num_utterances,
  sentiment,
  call_category,
  disposition,
  protocol_adherence,
  -- Rubric scores (10 criteria, 1-5 scale)
  CAST(get_json_object(clean_json, '$.greeting_score') AS INT) AS greeting_score,
  CAST(get_json_object(clean_json, '$.identity_verification_score') AS INT) AS identity_verification_score,
  CAST(get_json_object(clean_json, '$.empathy_score') AS INT) AS empathy_score,
  CAST(get_json_object(clean_json, '$.commitment_score') AS INT) AS commitment_score,
  CAST(get_json_object(clean_json, '$.branding_score') AS INT) AS branding_score,
  CAST(get_json_object(clean_json, '$.compliance_score') AS INT) AS compliance_score,
  CAST(get_json_object(clean_json, '$.resolution_score') AS INT) AS resolution_score,
  CAST(get_json_object(clean_json, '$.further_assistance_score') AS INT) AS further_assistance_score,
  CAST(get_json_object(clean_json, '$.closing_score') AS INT) AS closing_score,
  CAST(get_json_object(clean_json, '$.customer_service_score') AS INT) AS customer_service_score,
  CAST(get_json_object(clean_json, '$.overall_score') AS DOUBLE) AS overall_qa_score,
  ROUND(greeting_adherence_score, 3) AS greeting_adherence,
  ROUND(closing_adherence_score, 3) AS closing_adherence,
  -- Coaching
  get_json_object(clean_json, '$.coaching_notes') AS coaching_notes,
  CAST(get_json_object(clean_json, '$.requires_human_review') AS BOOLEAN) AS requires_human_review,
  -- AI-generated summary for dashboards (20 words)
  ai_summarize(full_transcript, 20) AS call_summary,
  -- HIPAA: PII-redacted transcript safe for sharing with QA vendors
  ai_mask(full_transcript, ARRAY('person', 'phone', 'address', 'ssn')) AS redacted_transcript,
  full_transcript,
  current_timestamp() AS evaluated_at
FROM cleaned;

-- Verify: show summary after table creation
SELECT 
  COUNT(*) AS total_scored,
  SUM(CASE WHEN greeting_score IS NOT NULL THEN 1 ELSE 0 END) AS scores_parsed_ok,
  ROUND(AVG(overall_qa_score), 2) AS avg_overall_qa,
  ROUND(AVG(greeting_adherence), 3) AS avg_greeting_adherence,
  SUM(CASE WHEN requires_human_review THEN 1 ELSE 0 END) AS flagged_for_review
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard

total_scored,scores_parsed_ok,avg_overall_qa,avg_greeting_adherence,flagged_for_review
50,50,4.31,0.77,2


In [0]:
%sql
-- Final gold table: one row per call with all QA scores + AI enrichment
SELECT 
  interaction_id,
  agent_name,
  queue,
  sentiment,
  call_category,
  disposition,
  protocol_adherence,
  overall_qa_score,
  greeting_score,
  identity_verification_score,
  empathy_score,
  compliance_score,
  resolution_score,
  greeting_adherence,
  closing_adherence,
  requires_human_review,
  call_summary,
  coaching_notes
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
ORDER BY overall_qa_score ASC
LIMIT 20

interaction_id,agent_name,queue,sentiment,call_category,disposition,protocol_adherence,overall_qa_score,greeting_score,identity_verification_score,empathy_score,compliance_score,resolution_score,greeting_adherence,closing_adherence,requires_human_review,call_summary,coaching_notes
cd79b388-70a6-42d8-bb8e-996a14032e86,Maria Garcia,Medical Records,negative,Billing Dispute,routine,non_compliant,1.5,4,1,1,1,1,0.786,0.759,true,Customer inquires about duplicate bill payment,"The agent failed to verify the caller's identity, did not acknowledge their concern with empathy, and did not provide a clear resolution to the issue."
e0db79e2-033a-4e5a-af06-c8d72c4360a5,Amy Martinez,Appointments,neutral,Appointment Scheduling,routine,compliant,1.55,4,1,1,1,1,0.832,0.8,true,Caller confirms appointment with Dr. Thompson.,"The agent failed to verify the caller's identity, did not demonstrate empathy or commitment to help, and did not provide a resolution to the caller's issue."
e2bd00a9-33a3-46e8-932e-f32b9f19de88,Kevin Rodriguez,Insurance Verification,neutral,Clinical Triage,routine,compliant,2.75,5,5,1,5,1,0.724,0.682,false,Carlos Garcia calls nurse advice line.,"The agent needs to improve on empathy, commitment, resolution, and closing the call, as the current interaction only covers greeting, identity verification, and compliance."
d8fb12a7-d28e-457d-bb66-8ba35c74791e,Carlos Kim,Insurance Verification,neutral,Clinical Triage,routine,compliant,3.05,5,5,1,5,1,0.725,0.6,false,Patient requests cardiologist referral assistance,"The agent needs to improve on empathy and acknowledgment, as well as providing a resolution or clear next steps for the caller's issue."
b2e3aa95-773f-433d-a217-1953d5f81f60,Kevin Rodriguez,Billing,neutral,Billing Dispute,routine,compliant,3.35,5,4,4,5,1,0.772,0.617,false,Robert Smith inquires about duplicate bill payment.,The agent should have provided a resolution or clear next steps to the caller's issue and ended the call with a warm closing.
34167287-fee3-4863-a446-38e27afb36f7,Kevin Rodriguez,Nurse Advice,neutral,Clinical Triage,routine,compliant,3.75,5,5,1,5,5,0.724,0.797,false,Nurse advises patient to see doctor due to high blood sugar.,The agent could improve by acknowledging the caller's concern with empathy and explicitly stating their willingness to help.
2f88a3f6-72d8-404b-ae42-79c45875a04b,Robert Park,Insurance Verification,neutral,Appointment Scheduling,routine,non_compliant,3.75,4,5,2,5,2,0.866,0.704,false,Michael Chen cancels Dr. Williams appointment,The agent could improve by providing more empathetic responses and ensuring the caller's issue is fully resolved before ending the call.
2d6d8349-cd0a-4ac3-bd7b-8073dc907e8a,Robert Nguyen,Nurse Advice,neutral,Clinical Triage,routine,compliant,3.85,5,5,1,5,5,0.74,0.797,false,Nurse advises Jennifer to rest and monitor headache symptoms.,The agent could improve by showing more empathy and explicitly stating their commitment to helping the caller.
70b4ebb7-7765-4302-be19-39b24bdcc97d,Robert Park,Pharmacy,neutral,Clinical Triage,routine,compliant,3.85,5,5,1,5,5,0.726,0.797,false,Nurse advises rest for chest tightness,The agent could improve by acknowledging the caller's concern with empathy and explicitly stating their willingness to help.
f376dd5f-0667-478c-bba1-8c4160aa7e23,Amy Martinez,Medical Records,neutral,Appointment Scheduling,routine,compliant,3.85,5,5,1,5,5,0.719,0.619,false,Patient seeks orthopedist referral assistance,The agent should work on acknowledging caller concerns with empathy and providing a warm closing to the call.


## Gold Layer Complete — AI-Scored QA Evaluations

**What we just built:** Every call now has:
* **Sentiment** — Customer emotional state (Positive/Negative/Neutral/Mixed)
* **Category** — Business classification (Billing, Clinical, Scheduling, etc.)
* **5 QA Scores** — Each criterion 1-5 (greeting, empathy, accuracy, escalation, compliance)
* **Overall Score** — Weighted composite
* **Coaching Notes** — AI-generated improvement recommendations
* **Human Review Flag** — Low-confidence calls flagged for supervisor review

**Business Impact:** A human QA reviewer takes ~45 minutes per call. This AI pipeline scores 50 calls in seconds — projected **98% reduction in QA review time** for routine calls.

**Next notebook →** `03_LLM_Judge_Evals` validates these AI scores against golden reference data.

---
### For the pod
* The `gold_scorecard` view is what powers the dashboard and Genie Space
* `requires_human_review = true` calls flow to the HITL triage queue
* In production, this runs as a scheduled Lakeflow job on new call batches